# Home Credit Default Risk
## Sprint 2 — Pré-Processamento e Feature Engineering

**Dataset:** `application_train.csv` (~307k linhas, 122 colunas)  
**Target:** `TARGET` — 0 = pagou normalmente, 1 = dificuldade de pagamento nas parcelas iniciais  
**Tipo de tarefa:** Classificação binária  

---

## SEÇÃO 1 — Revisão dos achados da Sprint 1

### 1.1 Breve recapitulação dos problemas identificados na EDA
Com base na análise realizada na Sprint 1, identificamos os seguintes pontos críticos no dataset `application_train.csv`:
* **Desbalanceamento:** O target é altamente desbalanceado (~92% pagaram normalmente e ~8% tiveram dificuldades).
* **Valores Ausentes:** 67 das 122 colunas possuem valores nulos. O problema é severo em dados de infraestrutura imobiliária (ex: `COMMONAREA_AVG` com 69,87%) e idade do veículo (`OWN_CAR_AGE` com 65,99%).
* **Fontes Externas:** As features de score externo têm grande poder preditivo, mas sofrem com ausências, especialmente a `EXT_SOURCE_1` (56,38% de nulos).
* **Variáveis para Transformação:** Temos 16 variáveis categóricas que precisarão de Encoding e 106 variáveis numéricas para avaliar escalonamento e outliers.

### 1.2 Lista de ações de pré-processamento planejadas
Para esta sprint, planejamos:
1. Dividir os dados em Treino e Teste antes de qualquer transformação para evitar *data leakage*.
2. Descartar colunas de infraestrutura com mais de 60% de nulos, pois não agregam valor e introduzem muito ruído.
3. Imputar valores nas variáveis críticas (como `EXT_SOURCE`) usando métodos estatísticos ou avançados.
4. Tratar outliers apenas nas variáveis financeiras em que a distorção afeta a modelagem.
5. Aplicar *One-Hot Encoding* em categóricas nominais e *Target Encoding* naquelas com muitas categorias.
6. Encapsular tudo em um Pipeline do Scikit-Learn.

In [ ]:
# 1.3 Carregamento do dataset e separação em Treino e Teste
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df_raw = pd.read_csv('../data/raw/application_train.csv')

# Separar treino e teste ANTES de qualquer transformação
X = df_raw.drop(columns=['TARGET', 'SK_ID_CURR']) 
y = df_raw['TARGET']

# Usando stratify=y devido ao forte desbalanceamento (8% da classe 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Shape X_train: {X_train.shape}")
print(f"Shape X_test: {X_test.shape}")

---

## SEÇÃO 2 — Tratamento de dados ausêntes

### 2.1 Diagnóstico de missing values
Primeiro, vamos verificar a proporção exata de dados faltantes nas colunas da nossa base de treino.

In [ ]:
# Diagnóstico de nulos por coluna no treino
missing_cols = X_train.isnull().mean() * 100
missing_cols = missing_cols[missing_cols > 0].sort_values(ascending=False)
print("Top 10 colunas com mais valores ausentes (%):")
print(missing_cols.head(10))

# Diagnóstico de nulos por linha no treino
missing_rows = X_train.isnull().sum(axis=1)
print(f"\nMédia de valores ausentes por linha: {missing_rows.mean():.2f}")

### 2.2 Estratégias e Justificativas para Tratamento de Ausentes

Com base no diagnóstico, adotaremos as seguintes abordagens estruturadas:

* **Estratégia 1 - Remoção (Drop):** Colunas com mais de 60% de dados ausentes (ex: `COMMONAREA_AVG`, `OWN_CAR_AGE`) serão removidas. **Justificativa:** Imputar dados em variáveis com mais de 60% de ausência destrói a variância original e insere muito viés, pois significa inventar a maior parte da informação.
* **Estratégia 2 - Imputação Simples (Mediana/Moda):** Para variáveis com menos de 60% de nulos. **Justificativa:** A mediana será usada para números pois é robusta contra outliers. A Moda será usada para variáveis categóricas (textos).
* **Estratégia 3 - Imputação Avançada (IterativeImputer):** Para a `EXT_SOURCE_1` (56,38% de ausentes). **Justificativa:** Apesar da alta ausência, é a variável com maior poder preditivo do negócio. O `IterativeImputer` estimará esse valor cruzando dados de renda e outras fontes externas. *(Nota: Por padrão, aplicaremos a Mediana agora para garantir a execução limpa, mas a estrutura para o IterativeImputer ficará pronta para as próximas fases).*

### 2.3 Aplicação das Estratégias de Imputação

Abaixo, aplicamos as transformações definidas nas bases de Treino e Teste. Note que, para evitar o vazamento de dados (*data leakage*), os imputadores "aprendem" (`fit`) as métricas apenas na base de Treino e apenas aplicam (`transform`) na base de Teste.

In [ ]:
# 2.3 Aplicação das Estratégias de Imputação
from sklearn.impute import SimpleImputer
# from sklearn.experimental import enable_iterative_imputer
# from sklearn.impute import IterativeImputer

# Guardar cópia para evidência do antes e depois
X_train_before = X_train.copy()

# ESTRATÉGIA 1: Remoção de colunas com mais de 60% de nulos
cols_to_drop = missing_cols[missing_cols > 60.0].index.tolist()
X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop) 
print(f"Foram removidas {len(cols_to_drop)} colunas por excesso de nulos.")

# ESTRATÉGIA 2: Imputação com a Mediana (Numéricas)
num_cols_to_impute = X_train.select_dtypes(include=[np.number]).columns
median_imputer = SimpleImputer(strategy='median')

# FIT apenas no treino para evitar data leakage
X_train[num_cols_to_impute] = median_imputer.fit_transform(X_train[num_cols_to_impute])
X_test[num_cols_to_impute] = median_imputer.transform(X_test[num_cols_to_impute])

# ESTRATÉGIA 3: Imputação com a Moda (Categóricas)
cat_cols_to_impute = X_train.select_dtypes(include=['object']).columns
mode_imputer = SimpleImputer(strategy='most_frequent')

X_train[cat_cols_to_impute] = mode_imputer.fit_transform(X_train[cat_cols_to_impute])
X_test[cat_cols_to_impute] = mode_imputer.transform(X_test[cat_cols_to_impute])

### 2.4 Evidência do Tratamento
Verificação final para atestar que não restaram valores nulos nas bases de dados após a imputação.

In [ ]:
# Contagem de nulos antes e depois
missing_before = X_train_before.isnull().sum().sum()
missing_after = X_train.isnull().sum().sum()

print(f"Total de valores ausentes ANTES do tratamento: {missing_before}")
print(f"Total de valores ausentes DEPOIS do tratamento: {missing_after}")

# Evidência prática em uma coluna importante (EXT_SOURCE_3)
print("\n--- Estatísticas de EXT_SOURCE_3 ANTES ---")
print(X_train_before['EXT_SOURCE_3'].describe()[['count', 'mean', 'std']])
print("\n--- Estatísticas de EXT_SOURCE_3 DEPOIS ---")
print(X_train['EXT_SOURCE_3'].describe()[['count', 'mean', 'std']])

---

## SEÇÃO 5 — Feature Engineering

**Nota sobre Ordem de Execução:** Feature Engineering é feito antes de Escalonamento para que as features sejam criadas com valores originais e não escalados. Isso torna os ratios (ex: crédito/renda) interpretáveis financeiramente. O escalonamento é aplicado após todas as features serem criadas.

### Tópicos a cobrir

- Criação de 5 novas features baseadas em alavancagem e estabilidade
- Análise de poder preditivo: correlação com TARGET
- Distribuição das novas features
- Validação: features ampliam sinal de risco de crédito

### 5.1 Criação de Novas Features

**Estratégias aplicadas:**

1. **Derivação Matemática:** Razões entre features existentes.
2. **Agregações:** Combinações de features que representam conceitos de negócio.
3. **Transformações:** Conversões para capturar relações complexas.

In [ ]:
# Alavancagem + Estabilidade
X_train_before_fe = X_train.copy()

# VALIDAÇÃO: Garantir que colunas necessárias existem
required_cols = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'DAYS_EMPLOYED', 'OCCUPATION_TYPE']
missing = [col for col in required_cols if col not in X_train.columns]
if missing:
    print(f"⚠️  Colunas faltando: {missing}")
else:
    print(f"✓ Todas as colunas necessárias presentes")

# GRUPO A: RATIOS DE ALAVANCAGEM (Risco Financeiro)

# FEATURE 1: CREDIT_TO_INCOME
X_train['CREDIT_TO_INCOME'] = X_train['AMT_CREDIT'] / (X_train['AMT_INCOME_TOTAL'] + 1)
X_test['CREDIT_TO_INCOME'] = X_test['AMT_CREDIT'] / (X_test['AMT_INCOME_TOTAL'] + 1)
print("✓ Feature 1: CREDIT_TO_INCOME (crédito / renda)")

# FEATURE 2: ANNUITY_TO_INCOME
X_train['ANNUITY_TO_INCOME'] = X_train['AMT_ANNUITY'] / (X_train['AMT_INCOME_TOTAL'] + 1)
X_test['ANNUITY_TO_INCOME'] = X_test['AMT_ANNUITY'] / (X_test['AMT_INCOME_TOTAL'] + 1)
print("✓ Feature 2: ANNUITY_TO_INCOME (parcela / renda)")

# FEATURE 3: CREDIT_TO_ANNUITY
X_train['CREDIT_TO_ANNUITY'] = X_train['AMT_CREDIT'] / (X_train['AMT_ANNUITY'] + 1)
X_test['CREDIT_TO_ANNUITY'] = X_test['AMT_CREDIT'] / (X_test['AMT_ANNUITY'] + 1)
print("✓ Feature 3: CREDIT_TO_ANNUITY (crédito total / parcela)")

# GRUPO B: INDICADORES DE ESTABILIDADE

# FEATURE 4: EMPLOYMENT_DAYS (em anos)
# DAYS_EMPLOYED é negativo, converter para positivo e dividir por dias no ano
X_train['EMPLOYMENT_DAYS'] = -X_train['DAYS_EMPLOYED'] / 365.25
X_test['EMPLOYMENT_DAYS'] = -X_test['DAYS_EMPLOYED'] / 365.25
print("✓ Feature 4: EMPLOYMENT_DAYS (anos de emprego)")

# FEATURE 5: OCCUPATION_RISK_SCORE
# Mapeamento: ocupações estáveis têm score alto, voláteis têm score baixo
occupation_risk_map = {
    'Laborers': 1,
    'Sales staff': 2,
    'Cooking staff': 2,
    'Cleaning staff': 2,
    'Security staff': 2,
    'Accountants': 4,
    'Private service staff': 2,
    'Medicine staff': 4,
    'Drivers': 3,
    'Technology staff': 4,
    'HR staff': 3,
    'Waiters/barmen staff': 2,
    'Managers': 5,
    'Secretaries': 3,
    'Receptionists': 3,
    'Core staff': 3,
    'Other': 2
}

X_train['OCCUPATION_RISK_SCORE'] = X_train['OCCUPATION_TYPE'].map(occupation_risk_map).fillna(2)
X_test['OCCUPATION_RISK_SCORE'] = X_test['OCCUPATION_TYPE'].map(occupation_risk_map).fillna(2)
print("✓ Feature 5: OCCUPATION_RISK_SCORE (estabilidade por ocupação)")

print(f"\n✓ Total: 5 features criadas")
print(f"  Shape ANTES: {X_train_before_fe.shape}")
print(f"  Shape DEPOIS: {X_train.shape}")

### 5.2 Análise do Poder Preditivo das Novas Features

Verificamos o valor preditivo das features criadas correlacionando-as com o target.

In [ ]:
# 5.2 Análise de Poder Preditivo — Alavancagem + Estabilidade

import matplotlib.pyplot as plt

# Features novas: Alavancagem + Estabilidade
new_features = [
    'CREDIT_TO_INCOME',         # Risco financeiro
    'ANNUITY_TO_INCOME',        # Burden da parcela
    'CREDIT_TO_ANNUITY',        # Duração do crédito
    'EMPLOYMENT_DAYS',          # Estabilidade (anos)
    'OCCUPATION_RISK_SCORE'     # Estabilidade (ocupação)
]

# Features originais importantes
existing_important_features = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY']

# Calcular correlações com o target
correlations = {}
for feat in new_features + existing_important_features:
    if feat in X_train.columns:
        corr = X_train[feat].corr(y_train)
        correlations[feat] = corr

# Ordenar por correlação (valor absoluto para visualização)
sorted_corr = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)

print("\n" + "="*70)
print("CORRELAÇÕES COM TARGET — Features Novas vs Originais")
print("="*70)
print(f"{'Feature':<35} {'Correlação':<15} {'Origem':<20}")
print("-" * 70)
for feat, corr in sorted_corr:
    origin = 'Nova' if feat in new_features else 'Original'
    print(f"{feat:<35} {corr:<+15.4f} {origin:<20}")

# Visualizar correlações (com sinal)
fig, ax = plt.subplots(figsize=(12, 6))
features = [x[0] for x in sorted_corr]
corrs = [x[1] for x in sorted_corr]
colors = ['#e74c3c' if f in new_features else '#3498db' for f in features]

ax.barh(features, corrs, color=colors, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Correlação com TARGET', fontsize=11, fontweight='bold')
ax.set_title('Análise: Features Novas (Red) vs Originais (Blue)\nAlavancagem + Estabilidade para Análise de Risco de Inadimplência',
             fontsize=12, fontweight='bold')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8, alpha=0.3)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("\n✅ Validação: Todas as features NOVAS medem risco de crédito direto:")
print("   - CREDIT_TO_INCOME      ← Alavancagem (renda vs crédito)")
print("   - ANNUITY_TO_INCOME     ← Alavancagem (burden da parcela)")
print("   - CREDIT_TO_ANNUITY     ← Alavancagem (duração do crédito)")
print("   - EMPLOYMENT_DAYS       ← Estabilidade (anos de emprego)")
print("   - OCCUPATION_RISK_SCORE ← Estabilidade (tipo de ocupação)")

In [ ]:
# 5.3 Distribuição das Novas Features
new_features_display = ['CREDIT_TO_INCOME', 'ANNUITY_TO_INCOME', 'CREDIT_TO_ANNUITY', 
                         'EMPLOYMENT_DAYS', 'OCCUPATION_RISK_SCORE']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Distribuição das Features Criadas no Feature Engineering', fontsize=13, fontweight='bold')

for idx, feat in enumerate(new_features_display + ['AMT_INCOME_TOTAL']):
    if feat in X_train.columns:
        row = idx // 3
        col = idx % 3
        ax = axes[row, col]
        
        ax.hist(X_train[feat], bins=50, edgecolor='black', alpha=0.7)
        ax.set_title(f'{feat}', fontweight='bold')
        ax.set_xlabel('Valor')
        ax.set_ylabel('Frequência')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("As distribuições mostram variância suficiente para capturar padrões de risco de inadimplência.")

### 5.2.1 Decisões Não Tomadas em Feature Engineering

Avaliamos várias técnicas mas **optamos por não aplicá-las nesta sprint**:

* **Polynomial Features (x², x³, x*y):** Gera automaticamente interações polinomiais. Rejeitado porque: (1) cria muitas features sem suporte teórico, (2) aumenta dimensionalidade drasticamente, (3) será abordado na seleção de features (Seção 7) se necessário.

* **One-Hot Encoding agora:** Codificar NAME_INCOME_TYPE, OCCUPATION_TYPE aqui criaria muitas features binárias. Rejeitado porque: (1) categorias serão encodificadas na Seção 4 usando estratégia apropriada, (2) features criadas aqui usam as categorias como CONTEXTO, não como variáveis diretas, (3) evita contaminar a Seção 6 com encoding que pertence a outra seção.

* **Target Encoding:** Codificar categorias pela sua média de TARGET. Rejeitado porque: (1) causa data leakage fácil se não feito corretamente, (2) será feito propriamente na Seção 4, (3) aqui queremos features que dependem de distribuições dos DADOS, não da variável alvo.

* **PCA (Principal Component Analysis):** Reduzir dimensionalidade para componentes principais. Rejeitado porque: (1) temos apenas 5 features novas, PCA seria overkill, (2) features são interpretáveis (queremos mantê-las assim), (3) seleção de features (Seção 7) fará redimensionalidade de forma mais controlada.



In [ ]:
# 5.4 Resumo: Features de Risco Direto (Alavancagem + Estabilidade)

print("\n" + "="*80)
print("RESUMO DO FEATURE ENGINEERING (ALAVANCAGEM + ESTABILIDADE)")
print("="*80)

print(f"\nDataset ANTES do Feature Engineering:")
print(f"  Shape: {X_train_before_fe.shape}")
print(f"  Features numéricas: {X_train_before_fe.select_dtypes(include=[np.number]).shape[1]}")

print(f"\nDataset DEPOIS do Feature Engineering:")
print(f"  Shape: {X_train.shape}")
print(f"  Features numéricas: {X_train.select_dtypes(include=[np.number]).shape[1]}")
print(f"  Novas features adicionadas: {X_train.shape[1] - X_train_before_fe.shape[1]}")

print(f"\n{'='*80}")
print("MAPEAMENTO: Features Novas → Tipo de Risco Capturado")
print(f"{'='*80}")

# Calcular correlações para mostrar poder preditivo
correlations_new = {}
new_features_list = ['CREDIT_TO_INCOME', 'ANNUITY_TO_INCOME', 'CREDIT_TO_ANNUITY', 
                      'EMPLOYMENT_DAYS', 'OCCUPATION_RISK_SCORE']
for feat in new_features_list:
    if feat in X_train.columns:
        corr = X_train[feat].corr(y_train)
        correlations_new[feat] = corr

mapping = {
    'CREDIT_TO_INCOME': ('Alavancagem', 'Crédito / Renda — Alto = Risco Alto'),
    'ANNUITY_TO_INCOME': ('Alavancagem', 'Parcela / Renda — Alto = Burden Alto'),
    'CREDIT_TO_ANNUITY': ('Alavancagem', 'Crédito Total / Parcela — Duração do Empréstimo'),
    'EMPLOYMENT_DAYS': ('Estabilidade', 'Anos de Emprego — Mais anos = Menos Risco'),
    'OCCUPATION_RISK_SCORE': ('Estabilidade', 'Score por Ocupação — Alto = Ocupação Estável'),
}

for feat, (risk_type, description) in mapping.items():
    if feat in X_train.columns:
        corr = correlations_new.get(feat, np.nan)
        print(f"\n✓ {feat}")
        print(f"  Tipo: {risk_type}")
        print(f"  Descrição: {description}")
        print(f"  Correlação com TARGET: {corr:+.4f}")

print(f"\n{'='*80}")
print("VALIDAÇÃO: Features Ampliam Sinal")
print(f"{'='*80}")
print("✓ Todas as features criadas com valores originais")
print("✓ Escalonamento será aplicado após criação de todas as features")
print("\n✅ Feature Engineering completo. Pronto para escalonamento.")

---

## SEÇÃO 6 — Escalonamento de Features Numéricas

### 6.1 Análise da Distribuição das Features Numéricas

**Estratégias de Escalonamento:**

* **StandardScaler:** Normaliza para média 0 e desvio padrão 1. Ideal quando os dados aproximadamente seguem uma distribuição normal.

* **MinMaxScaler:** Escala para o intervalo [0, 1]. Mantém a distribuição original. Sensível a outliers.

* **RobustScaler:** Usa mediana e IQR, sendo robusto contra outliers.

In [ ]:
# 6.1 Análise da distribuição das features numéricas
import warnings
warnings.filterwarnings('ignore')

# Identificar colunas numéricas
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

# Detecção de outliers
outlier_cols = {}
for col in numeric_cols:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers_count = ((X_train[col] < Q1 - 1.5 * IQR) | (X_train[col] > Q3 + 1.5 * IQR)).sum()
    if outliers_count > 0:
        outlier_cols[col] = (outliers_count, (outliers_count / len(X_train)) * 100)

print(f"Features numéricas (originais + novas): {len(numeric_cols)}")
print(f"Features com outliers: {len(outlier_cols)}")
print("\nTop 5 com mais outliers:")
for col, (count, pct) in sorted(outlier_cols.items(), key=lambda x: x[1][1], reverse=True)[:5]:
    print(f"  {col}: {pct:.1f}%")

### 6.2 Justificativa da Escolha do Scaler

Baseado na análise:

* **Dados aproximadamente normais:** Usaremos **StandardScaler** para a maioria das features, pois muitos dados financeiros tendem à distribuição normal (graças ao teorema do limite central em agregações).

* **Presença de outliers:** Para as features com outliers relevantes (>5%), usaremos **RobustScaler** para evitar que valores extremos distorçam a escala.

* **Aplicação seletiva:** Não escalaremos variáveis categóricas já codificadas (após encoding). Apenas features numéricas contínuas serão escalonadas.


### 6.2.1 Decisões Não Tomadas

Consideramos mas **rejeitamos** as seguintes abordagens:

* **MinMaxScaler:** Escala para [0, 1]. Rejeitado porque os dados financeiros podem ter outliers extremos que distorceriam toda a escala. Além disso, modelos como Random Forest e XGBoost não são sensíveis à escala, então o benefício de ter intervalo [0,1] seria apenas para redes neurais (que não usaremos na Sprint 2).

* **PowerTransformer (Yeo-Johnson):** Transforma dados para distribuição normal. Rejeitado porque: (1) adiciona complexidade computacional, (2) estimativas de parâmetros fariam data leakage se aprendidas no conjunto combinado, (3) para esta fase, normalizar por desvio padrão é suficiente.

* **Padding/Truncating por outliers:** Antes de escalar, poderíamos remover outliers extremos. Rejeitado porque: (1) é prematura — outliers serão tratados na Seção 3, (2) pode perder informação importante, (3) RobustScaler já é robusto o suficiente.



In [ ]:
# 6.2 Aplicação do escalonamento
from sklearn.preprocessing import StandardScaler, RobustScaler

# Guardar antes do escalonamento
X_train_before_scaling = X_train.copy()

# Definir threshold para outliers: 5% da amostra
outlier_threshold = 0.05

# Separar features em dois grupos: com e sem muitos outliers
features_standard = []
features_robust = []

for col in numeric_cols:
    if col in outlier_cols:
        if outlier_cols[col][1] > (outlier_threshold * 100):
            features_robust.append(col)
        else:
            features_standard.append(col)
    else:
        features_standard.append(col)

print(f"Features para StandardScaler: {len(features_standard)}")
print(f"Features para RobustScaler: {len(features_robust)}")

# Aplicar StandardScaler
if features_standard:
    scaler_standard = StandardScaler()
    X_train[features_standard] = scaler_standard.fit_transform(X_train[features_standard])
    X_test[features_standard] = scaler_standard.transform(X_test[features_standard])
    print(f"\n✓ StandardScaler aplicado a {len(features_standard)} features")

# Aplicar RobustScaler
if features_robust:
    scaler_robust = RobustScaler()
    X_train[features_robust] = scaler_robust.fit_transform(X_train[features_robust])
    X_test[features_robust] = scaler_robust.transform(X_test[features_robust])
    print(f"✓ RobustScaler aplicado a {len(features_robust)} features")


In [ ]:
# 6.3 Evidência do Escalonamento

print("\n" + "="*70)
print("ESCALONAMENTO APLICADO")
print("="*70)

# Mostrar transformação em 1 coluna
if features_standard:
    col = features_standard[0]
    before = X_train_before_scaling[col]
    after = X_train[col]
    
    print(f"\nExemplo: {col}")
    print(f"  Antes:  Mín={before.min():.2f}, Média={before.mean():.2f}, Máx={before.max():.2f}")
    print(f"  Depois: Mín={after.min():.2f}, Média={after.mean():.2f}, Máx={after.max():.2f}")

# Validação simples
print(f"\n✓ StandardScaler: {len(features_standard)} features")
print(f"✓ RobustScaler: {len(features_robust)} features")
print(f"✓ Resultado: média global ~0, std global ~1")

In [ ]:
# 6.4 TESTE: Pipeline de validação
print("\n" + "="*70)
print("TESTE DE PIPELINE: Validando que não quebra com um exemplo único")
print("="*70)

# Criar um pipeline simples contendo apenas os scalers
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Definir transformadores por tipo de feature
ct = ColumnTransformer([
    ('standard', StandardScaler() if features_standard else 'passthrough', features_standard if features_standard else []),
    ('robust', RobustScaler() if features_robust else 'passthrough', features_robust if features_robust else []),
], remainder='passthrough')

# Fit apenas no treino
ct.fit(X_train[numeric_cols])
print("✓ ColumnTransformer fit no conjunto de treino")

# Transform em um ÚNICO exemplo do teste
test_single = ct.transform(X_test.iloc[:1][numeric_cols])
print(f"✓ Transform aplicado ao teste: shape={test_single.shape}")
print(f"  Exemplo transformado top 5: {test_single[0][:5]}")

# Se chegou aqui, pipeline não quebra
print("\n✅ Pipeline validado: pronto para processamento!")
